In [ ]:
!pip install groq
!pip install gradio
!pip install langchain
!pip install langchain-community
!pip install gtts
!pip install faiss-cpu
!pip install transformers==4.35.0
!pip install sentence-transformers
!pip install yt-dlp openai-whisper
!pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.3/469.3 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.79
    Uninstalling langchain-core-0.3.79:
      Successfully uninstalled langchain-core-0.3.79
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.11
    Uninstalling langchain-text-splitters-0.3.11:
      Successfu

In [ ]:
!pip install youtube-transcript-api


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.1/485.1 kB 15.1 MB/s eta 0:00:00


In [ ]:
import os
import re
import tempfile
import whisper
import yt_dlp
import gradio as gr
from groq import Groq
from gtts import gTTS
from youtube_transcript_api import YouTubeTranscriptApi
from sentence_transformers import SentenceTransformer
import faiss

# === Step 1: Extract YouTube Transcript ===
def get_transcript(youtube_url):
    ydl_opts = {"format": "bestaudio", "outtmpl": "temp_audio.%(ext)s"}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=True)
        audio_file = ydl.prepare_filename(info)

    model = whisper.load_model("base")  # or use "tiny" for faster processing
    result = model.transcribe(audio_file)
    return result["text"]

# === Step 2: Summarize & Clean Transcript ===
client = Groq(api_key="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")  # Set GROQ_API_KEY in your environment

def summarize_and_clean(text: str) -> str:
    # Clean
    clean_response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a transcript cleaner. Extract only the core informative content from the transcript."},
            {"role": "user", "content": f"Clean this transcript:\n\n{text}"}
        ]
    )
    cleaned = clean_response.choices[0].message.content.strip()
    os.makedirs("data", exist_ok=True)
    with open("data/cleaned.txt", "w", encoding="utf-8") as f:
        f.write(cleaned)

    # Summarize
    summary_response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "Write a podcast-style summary that captures the core ideas in an engaging way."},
            {"role": "user", "content": f"Create podcast script:\n\n{cleaned}"}
        ]
    )
    summary = summary_response.choices[0].message.content.strip()
    with open("data/summary.txt", "w", encoding="utf-8") as f:
        f.write(summary)
    return summary

# === Step 3: RAG Indexing ===
def build_vector_index():
    with open("data/cleaned.txt", "r", encoding="utf-8") as f:
        text = f.read()
    chunks = [text[i:i+500] for i in range(0, len(text), 500)]
    model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = model.encode(chunks)
    index = faiss.IndexFlatL2(embeddings[0].shape[0])
    index.add(embeddings)
    faiss.write_index(index, "data/faiss.index")
    with open("data/chunks.txt", "w", encoding="utf-8") as f:
        f.write("\n---SPLIT---\n".join(chunks))

# === Step 4: QA via RAG ===
def answer_question(query: str) -> str:
    model = SentenceTransformer("all-MiniLM-L6-v2")
    index = faiss.read_index("data/faiss.index")
    with open("data/chunks.txt", "r", encoding="utf-8") as f:
        chunks = f.read().split("\n---SPLIT---\n")
    query_vec = model.encode([query])
    D, I = index.search(query_vec, k=3)
    context = "\n\n".join([chunks[i] for i in I[0]])
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a helpful assistant answering questions based on transcript content."},
            {"role": "user", "content": f"Transcript:\n{context}\n\nQuestion:\n{query}"}
        ]
    )
    return response.choices[0].message.content.strip()

# === Step 5: Podcast Audio ===
def generate_audio(script: str) -> str:
    tts = gTTS(script)
    path = tempfile.mktemp(suffix=".mp3")
    tts.save(path)
    return path

# === Step 6: Gradio UI ===
def process_video(url):
    try:
        raw = get_transcript(url)
        summary = summarize_and_clean(raw)
        build_vector_index()
        return summary
    except Exception as e:
        return f"Error: {e}"

def rag_query(q):
    try:
        return answer_question(q)
    except Exception as e:
        return f"Error: {e}"

def podcast_audio():
    try:
        with open("data/summary.txt", "r", encoding="utf-8") as f:
            script = f.read()
        return generate_audio(script)
    except Exception as e:
        return f"Error: {e}"

with gr.Blocks(title="YouTube RAG Podcast") as demo:
    gr.Markdown("## 🎙️ YouTube to RAG-Podcast Converter")
    url_input = gr.Textbox(label="Paste YouTube Link")
    summary_output = gr.Textbox(label="Podcast Script Summary")
    btn1 = gr.Button("Generate Summary + Index")
    btn1.click(fn=process_video, inputs=url_input, outputs=summary_output)

    gr.Markdown("### Ask a Question")
    query_input = gr.Textbox(label="Your Question")
    answer_output = gr.Textbox(label="Answer")
    btn2 = gr.Button("Ask")
    btn2.click(fn=rag_query, inputs=query_input, outputs=answer_output)

    gr.Markdown("### 🎧 Generate Podcast Audio")
    audio_output = gr.Audio(label="Podcast", type="filepath")
    btn3 = gr.Button("Generate Audio")
    btn3.click(fn=podcast_audio, outputs=audio_output)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1261c45b5d801cef13.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install -U yt-dlp

In [ ]:
import os
import tempfile
import whisper
import yt_dlp
import gradio as gr
from groq import Groq
from gtts import gTTS
from sentence_transformers import SentenceTransformer
from deep_translator import GoogleTranslator
import faiss
from youtube_transcript_api import YouTubeTranscriptApi

# === Step 1: Extract YouTube Transcript ===
def get_transcript(youtube_url):
    try:
        video_id = youtube_url.split("v=")[1].split("&")[0]
        transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
        transcript = transcript_list.find_transcript(['en', 'a.en']) # Try English first, then auto-generated English
        return " ".join([d['text'] for d in transcript.fetch()])
    except Exception as e:
        # Fallback to downloading audio and transcribing if transcript API fails
        print(f"Transcript API failed: {e}. Falling back to audio download.")
        ydl_opts = {"format": "bestaudio", "outtmpl": "temp_audio.%(ext)s"}
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(youtube_url, download=True)
            audio_file = ydl.prepare_filename(info)

        model = whisper.load_model("base")
        result = model.transcribe(audio_file)
        return result["text"]


# === Step 2: Summarize & Clean Transcript ===
client = Groq(api_key="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")  # Replace with your key

def summarize_and_clean(text: str) -> str:
    # Clean transcript
    clean_response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a transcript cleaner. Extract only core informative content."},
            {"role": "user", "content": f"Clean this transcript:\n\n{text}"}
        ]
    )
    cleaned = clean_response.choices[0].message.content.strip()
    os.makedirs("data", exist_ok=True)
    with open("data/cleaned.txt", "w", encoding="utf-8") as f:
        f.write(cleaned)

    audio_summary = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": (
                "Write a continuous podcast-style summary from the following content. "
                "The output should be 100-120 words long, in a single paragraph, "
                "and ideal for audio narration. Avoid headings or bullet points."
            )},
            {"role": "user", "content": f"Generate long podcast paragraph from:\n\n{cleaned}"}
        ]
    )
    aud_sum = audio_summary.choices[0].message.content.strip()
    with open("data/audio_summary.txt", "w", encoding="utf-8") as f:
        f.write(aud_sum)

    summary_response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "Write a podcast-style summary that captures the core ideas in an engaging way."},
            {"role": "user", "content": f"Create podcast script:\n\n{cleaned}"}
        ]
    )
    summary = summary_response.choices[0].message.content.strip()
    with open("data/summary.txt", "w", encoding="utf-8") as f:
        f.write(summary)
    return summary

# === Step 3: RAG Indexing ===
def build_vector_index():
    with open("data/cleaned.txt", "r", encoding="utf-8") as f:
        text = f.read()
    chunks = [text[i:i+500] for i in range(0, len(text), 500)]
    model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = model.encode(chunks)
    index = faiss.IndexFlatL2(embeddings[0].shape[0])
    index.add(embeddings)
    faiss.write_index(index, "data/faiss.index")
    with open("data/chunks.txt", "w", encoding="utf-8") as f:
        f.write("\n---SPLIT---\n".join(chunks))

# === Step 4: QA via RAG ===
def answer_question(query: str) -> str:
    model = SentenceTransformer("all-MiniLM-L6-v2")
    index = faiss.read_index("data/faiss.index")
    with open("data/chunks.txt", "r", encoding="utf-8") as f:
        chunks = f.read().split("\n---SPLIT---\n")
    query_vec = model.encode([query])
    D, I = index.search(query_vec, k=3)
    context = "\n\n".join([chunks[i] for i in I[0]])
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a helpful assistant answering questions based on transcript content."},
            {"role": "user", "content": f"Transcript:\n{context}\n\nQuestion:\n{query}"}
        ]
    )
    return response.choices[0].message.content.strip()

# === Step 5: Multilingual Podcast Audio ===
def generate_audio(script: str, target_lang: str = "en") -> str:
    try:
        if target_lang != "en":
            script = GoogleTranslator(source="auto", target=target_lang).translate(script)
        tts = gTTS(script, lang=target_lang)
        path = tempfile.mktemp(suffix=".mp3")
        tts.save(path)
        return path
    except Exception as e:
        return f"Error generating audio: {e}"

def podcast_audio(language_code="en"):
    try:
        with open("data/audio_summary.txt", "r", encoding="utf-8") as f:
            script = f.read()
        return generate_audio(script, target_lang=language_code)
    except Exception as e:
        return f"Error: {e}"

# === Step 6: Gradio UI ===
def process_video(url):
    try:
        raw = get_transcript(url)
        summary = summarize_and_clean(raw)
        build_vector_index()
        return summary
    except Exception as e:
        return f"Error: {e}"

def rag_query(q):
    try:
        return answer_question(q)
    except Exception as e:
        return f"Error: {e}"

with gr.Blocks(title="YouTube RAG Podcast") as demo:
    gr.Markdown("## 🎙️ YouTube to Multilingual Podcast RAG App")

    url_input = gr.Textbox(label="Paste YouTube Link")
    summary_output = gr.Textbox(label="Podcast Script Summary")
    btn1 = gr.Button("Generate Summary")
    btn1.click(fn=process_video, inputs=url_input, outputs=summary_output)

    gr.Markdown("### 🔍 Ask a Question")
    query_input = gr.Textbox(label="Your Question")
    answer_output = gr.Textbox(label="Answer")
    btn2 = gr.Button("Ask")
    btn2.click(fn=rag_query, inputs=query_input, outputs=answer_output)

    gr.Markdown("### 🎧 Generate Multilingual Podcast Audio")
    lang_input = gr.Textbox(label="Language Code (e.g., en, hi, fr, es)", value="en")
    audio_output = gr.Audio(label="Podcast Audio", type="filepath")
    btn3 = gr.Button("Generate Audio")
    btn3.click(fn=podcast_audio, inputs=lang_input, outputs=audio_output)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://38325fe4ce1e2433ea.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import yt_dlp

def download_audio(url):
    ydl_opts = {
        'format': 'bestaudio/best',
        'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'http_headers': {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
            'Accept-Language': 'en-US,en;q=0.9',
        },
        'outtmpl': '%(title)s.%(ext)s',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])
# Example
download_audio("https://youtu.be/FwOTs4UxQS4")


[youtube] Extracting URL: https://youtu.be/FwOTs4UxQS4
[youtube] FwOTs4UxQS4: Downloading webpage
[youtube] FwOTs4UxQS4: Downloading tv client config
[youtube] FwOTs4UxQS4: Downloading tv player API JSON
[youtube] FwOTs4UxQS4: Downloading ios player API JSON
[youtube] FwOTs4UxQS4: Downloading m3u8 information
[info] FwOTs4UxQS4: Downloading 1 format(s): 251


ERROR: unable to download video data: HTTP Error 403: Forbidden


DownloadError: ERROR: unable to download video data: HTTP Error 403: Forbidden

In [ ]:
!pip install pytube


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.4 MB/s eta 0:00:00


In [ ]:
from pytube import YouTube

def download_audio(url):
    yt = YouTube(url)
    audio_stream = yt.streams.filter(only_audio=True).first()
    audio_stream.download()

download_audio("https://www.youtube.com/watch?v=FwOTs4UxQS4")


HTTPError: HTTP Error 400: Bad Request